In [1]:
# | echo: false

import os
from pathlib import Path
import re
from typing import Dict, List, Set

import duckdb
from dotenv import load_dotenv
import pandas as pd
import numpy as np

In [2]:
# | echo: false

def infer_table_order(schema_file_path: str) -> List[str]:
    """
    Infer the order of table creation based on foreign key dependencies.

    This function reads a SQL schema file, analyzes the CREATE TABLE statements
    and their foreign key relationships, and returns a list of table names in an
    order that ensures all dependencies are met when creating the tables.

    The order starts with tables that have no dependencies (top-level tables) and
    ends with tables that depend on others.

    Args:
        schema_file_path (str): The file path to the SQL schema file.

    Returns:
        List[str]: A list of table names in the order they should be created,
                   starting with tables with no dependencies.

    Raises:
        FileNotFoundError: If the specified schema file cannot be found.
        ValueError: If no valid CREATE TABLE statements are found in the file.
    """
    # Read the schema from the file
    try:
        with open(schema_file_path, "r") as file:
            schema_sql = file.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"Schema file not found: {schema_file_path}")

    # Remove comments and split into CREATE TABLE statements
    schema_sql = remove_comments(schema_sql)
    create_statements = split_create_statements(schema_sql)

    if not create_statements:
        raise ValueError("No valid CREATE TABLE statements found in the schema file.")

    # Build dependency graph
    dependencies = build_dependency_graph(create_statements)

    # Perform topological sort
    sorted_tables = topological_sort(dependencies)

    return sorted_tables  # This is already in the desired order (top-level first)


# | echo: false

def remove_comments(sql: str) -> str:
    """
    Remove SQL comments from the input string.

    Args:
        sql (str): The input SQL string.

    Returns:
        str: The SQL string with comments removed.
    """
    return re.sub(r"--.*$", "", sql, flags=re.MULTILINE)


def split_create_statements(sql: str) -> List[str]:
    """
    Split the SQL string into individual CREATE TABLE statements.

    Args:
        sql (str): The input SQL string.

    Returns:
        List[str]: A list of CREATE TABLE statements.
    """
    return [stmt.strip() for stmt in re.split(r"(?i)CREATE\s+OR\s+REPLACE\s+TABLE", sql)[1:]]


def build_dependency_graph(create_statements: List[str]) -> Dict[str, Set[str]]:
    """
    Build a graph of table dependencies based on foreign key relationships.

    Args:
        create_statements (List[str]): A list of CREATE TABLE statements.

    Returns:
        Dict[str, Set[str]]: A dictionary where keys are table names and values
                             are sets of tables they depend on.
    """
    dependencies: Dict[str, Set[str]] = {}

    for statement in create_statements:
        match = re.match(r"\s*(\w+)", statement)
        if match:
            current_table = match.group(1)
            dependencies[current_table] = set()

            fk_references = re.findall(r"FOREIGN KEY.*REFERENCES\s+(\w+)", statement, re.IGNORECASE)
            for referenced_table in fk_references:
                dependencies[current_table].add(referenced_table)

    return dependencies


def topological_sort(graph: Dict[str, Set[str]]) -> List[str]:
    """
    Perform a topological sort on the dependency graph.

    This implementation uses depth-first search to produce a topological ordering.
    The result is reversed to get the desired order (tables with no dependencies first).

    Args:
        graph (Dict[str, Set[str]]): The dependency graph.

    Returns:
        List[str]: A list of table names in topological order.
    """
    visited: Set[str] = set()
    result: List[str] = []

    def dfs(node: str) -> None:
        if node in visited:
            return
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor)
        result.append(node)

    for node in graph:
        if node not in visited:
            dfs(node)

    return result  # No need to reverse, this is already in the desired order

In [3]:
SCHEMA = Path("../sql/create_cb_experiment_tables_ddb.sql")

In [4]:
# | echo: false

try:
    table_order = infer_table_order(str(SCHEMA))
    print("Inferred table order (top-level first):", table_order)
except (FileNotFoundError, ValueError) as e:
    print(f"Error: {e}")

Inferred table order (top-level first): ['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']


In [5]:
# | echo: false

def copy_database_to_motherduck(source_db_path, md_db_name, tables=None):
    conn = duckdb.connect(":memory:")
    print('duckdb.connect(":memory:")')

    load_dotenv()

    md_token = os.getenv("MD_TOKEN")
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")
    else:
        print("`motherduck_token` environment variable set...")

    print("ATTACH 'md:';")
    conn.sql("ATTACH 'md:';")

    try:
        # Attach the source database
        print(f"ATTACH '{source_db_path}' AS local_source_db")
        conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

        print("USE 'local_source_db'")
        conn.execute("USE 'local_source_db'")

        # Get the list of tables from the source database
        if tables is None:
            tables = [table[0] for table in conn.execute("SHOW TABLES").fetchall()]

        # Create the new database
        print(f"CREATE OR REPLACE DATABASE {md_db_name}")
        conn.execute(f"CREATE OR REPLACE DATABASE {md_db_name}")

        # Copy the schema from the source database to the new database
        print(f"COPY FROM DATABASE local_source_db TO {md_db_name} (SCHEMA)")
        conn.execute(f"COPY FROM DATABASE local_source_db TO {md_db_name} (SCHEMA)")

        # Copy data for each table
        print(f"Table order: {tables}")
        for table_name in tables:
            print(f"INSERT INTO {md_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}")
            conn.execute(
                f"INSERT INTO {md_db_name}.{table_name} SELECT * FROM local_source_db.{table_name}"
            )

        print(f"\nINFO: Database '{md_db_name}' created and populated successfully.")

    except Exception as e:
        print(f"ERROR: An error occurred: {str(e)}")

    finally:
        conn.close()

    return md_db_name

In [6]:
# | echo: false

CB_DATABASE = Path("../data/Experiments_CB_2023_Q4.ddb")
CB_DATABASE_MD = str(CB_DATABASE.stem)

In [7]:
# | echo: false

# Turned off upload of database for now

md_db = copy_database_to_motherduck(CB_DATABASE, CB_DATABASE_MD, table_order)

duckdb.connect(":memory:")
`motherduck_token` environment variable set...
ATTACH 'md:';


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ATTACH '../data/Experiments_CB_2023_Q4.ddb' AS local_source_db
USE 'local_source_db'
CREATE OR REPLACE DATABASE Experiments_CB_2023_Q4


COPY FROM DATABASE local_source_db TO Experiments_CB_2023_Q4 (SCHEMA)


Table order: ['ExperimentCBs', 'TrialCBs', 'ResponseCBs', 'EventCBs']
INSERT INTO Experiments_CB_2023_Q4.ExperimentCBs SELECT * FROM local_source_db.ExperimentCBs


INSERT INTO Experiments_CB_2023_Q4.TrialCBs SELECT * FROM local_source_db.TrialCBs


INSERT INTO Experiments_CB_2023_Q4.ResponseCBs SELECT * FROM local_source_db.ResponseCBs


INSERT INTO Experiments_CB_2023_Q4.EventCBs SELECT * FROM local_source_db.EventCBs



INFO: Database 'Experiments_CB_2023_Q4' created and populated successfully.


In [8]:
# | echo: false

def compare_databases(local_db: str, md_db: str, threshold: float = 1e-8) -> bool:
    """
    Compare a local DuckDB database with a MotherDuck database.

    Args:
        local_db (str): Path to the local DuckDB database file.
        md_db (str): Name of the MotherDuck database.

    Returns:
        bool: True if databases are identical, False otherwise.

    Raises:
        ValueError: If MotherDuck token is not found in the .env file.
    """
    print(f"Comparing - LOCAL: {local_db} vs MOTHERDUCK: {md_db}")

    load_dotenv()
    md_token = os.getenv("MD_TOKEN")
    os.environ["motherduck_token"] = md_token
    if not md_token:
        raise ValueError("MotherDuck token not found in .env file")

    conn_md = duckdb.connect(f"md:{md_db}?motherduck_token={md_token}")
    conn_local = duckdb.connect(local_db)

    try:
        tables_local = conn_local.execute("SHOW TABLES").fetchall()
        tables_md = conn_md.execute("SHOW TABLES").fetchall()

        if len(tables_local) != len(tables_md):
            print(f"Number of tables mismatch. Local: {len(tables_local)}, MotherDuck: {len(tables_md)}")
            return False

        for table in tables_local:
            table_name = table[0]
            print(f"\nComparing table: {table_name}")

            if not compare_schema(conn_local, conn_md, table_name):
                return False

            if not compare_row_counts(conn_local, conn_md, table_name):
                return False

            if not compare_data(conn_local, conn_md, table_name):
                return False

        print("\nAll tables and data match!")
        return True

    finally:
        conn_local.close()
        conn_md.close()


def compare_schema(
    conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str
) -> bool:
    """
    Compare the schema of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        bool: True if schemas match, False otherwise.
    """
    schema_local = conn_local.execute(f"DESCRIBE {table_name}").fetchall()
    schema_md = conn_md.execute(f"DESCRIBE {table_name}").fetchall()

    if schema_local != schema_md:
        print(f"Schema mismatch for table {table_name}")
        print("Local schema:")
        print(pd.DataFrame(schema_local, columns=["Column", "Type", "Null", "Key", "Default", "Extra"]))
        print("\nMotherDuck schema:")
        print(pd.DataFrame(schema_md, columns=["Column", "Type", "Null", "Key", "Default", "Extra"]))
        return False
    return True


def compare_row_counts(
    conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str
) -> bool:
    """
    Compare the row counts of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        bool: True if row counts match, False otherwise.
    """
    count_local = conn_local.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    count_md = conn_md.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    if count_local != count_md:
        print(f"Row count mismatch for table {table_name}. Local: {count_local}, MotherDuck: {count_md}")
        return False
    return True


def compare_data(
    conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str, threshold: float = 1e-8
) -> bool:
    """
    Compare the data of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        bool: True if data matches, False otherwise.
    """
    schema_local = conn_local.execute(f"DESCRIBE {table_name}").fetchall()
    column_names = [col[0] for col in schema_local]

    order_by_clause = f"ORDER BY {', '.join(column_names)}"

    data_local = conn_local.execute(f"SELECT * FROM {table_name} {order_by_clause}").fetchall()
    data_md = conn_md.execute(f"SELECT * FROM {table_name} {order_by_clause}").fetchall()

    df_local = pd.DataFrame(data_local, columns=column_names)
    df_md = pd.DataFrame(data_md, columns=column_names)

    if not df_local.equals(df_md):
        print(f"Data mismatch for table {table_name}")

        for column in df_local.columns:
            if not df_local[column].equals(df_md[column]):
                print(f"\nMismatch in column: {column}")
                print(f"Local dtype: {df_local[column].dtype}, MotherDuck dtype: {df_md[column].dtype}")

                # Check for floating point differences
                if np.issubdtype(df_local[column].dtype, np.number) and np.issubdtype(
                    df_md[column].dtype, np.number
                ):
                    max_diff = (df_local[column] - df_md[column]).abs().max()
                    print(f"Maximum absolute difference: {max_diff}")

                    if max_diff < threshold:  # Adjust this threshold as needed
                        print("Difference likely due to floating point precision")
                    else:
                        print("First few mismatching rows:")
                        mismatch = ~np.isclose(df_local[column], df_md[column], equal_nan=True)
                        print(
                            pd.concat(
                                [df_local[mismatch], df_md[mismatch]],
                                axis=1,
                                keys=["Local", "MotherDuck"],
                            ).head()
                        )
                else:
                    print("First few mismatching rows:")
                    mismatch = df_local[column] != df_md[column]
                    print(
                        pd.concat(
                            [df_local[mismatch], df_md[mismatch]], axis=1, keys=["Local", "MotherDuck"]
                        ).head()
                    )

        # Check for rows that exist in one dataset but not the other
        merge_result = pd.merge(df_local, df_md, how="outer", indicator=True)
        local_only = merge_result[merge_result["_merge"] == "left_only"]
        md_only = merge_result[merge_result["_merge"] == "right_only"]

        if not local_only.empty:
            print("\nRows in Local but not in MotherDuck:")
            print(local_only.head())

        if not md_only.empty:
            print("\nRows in MotherDuck but not in Local:")
            print(md_only.head())

        if local_only.empty and md_only.empty:
            print(
                "\nNo rows found exclusively in either database. The mismatch might be due to data type or precision differences."
            )

        return False
    return True

In [9]:
# | echo: false

databases_identical = compare_databases(CB_DATABASE, CB_DATABASE_MD, threshold=1e-18)
print(f"Databases are identical: {databases_identical}")

Comparing - LOCAL: ../data/Experiments_CB_2023_Q4.ddb vs MOTHERDUCK: Experiments_CB_2023_Q4



Comparing table: EventCBs



Comparing table: ExperimentCBs



Comparing table: ResponseCBs



Comparing table: TrialCBs



All tables and data match!
Databases are identical: True


In [10]:
# | echo: false

def compare_table_schema(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str) -> pd.DataFrame:
    """
    Compare the schema of a table between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        pd.DataFrame: A DataFrame showing any differences in schema.
    """
    query = f"""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position
    """
    
    df_local = conn_local.execute(query).df()
    df_md = conn_md.execute(query).df()
    
    df_diff = pd.concat([df_local, df_md]).drop_duplicates(keep=False)
    return df_diff

def check_numeric_precision(conn: duckdb.DuckDBPyConnection, local_db_name: str, md_db_name: str, table_name: str, numeric_columns: list) -> dict:
    """
    Check for precision differences in numeric columns.

    Args:
        conn (duckdb.DuckDBPyConnection): Connection to the database.
        local_db_name (str): Name of the local database attachment.
        md_db_name (str): Name of the MotherDuck database attachment.
        table_name (str): Name of the table to compare.
        numeric_columns (list): List of numeric column names to check.

    Returns:
        dict: A dictionary with column names as keys and maximum differences as values.
    """
    results = {}
    for column in numeric_columns:
        query = f"""
        SELECT MAX(ABS(local.{column} - md.{column})) as max_diff
        FROM {local_db_name}.{table_name} as local
        JOIN {md_db_name}.{table_name} as md ON local.TrialID = md.TrialID
        """
        print(f"Executing query for column {column}:")
        print(query)
        max_diff = conn.execute(query).fetchone()[0]
        results[column] = max_diff
    return results

def compare_timestamps(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str, timestamp_column: str) -> pd.DataFrame:
    """
    Compare timestamp handling between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.
        timestamp_column (str): Name of the timestamp column to check.

    Returns:
        pd.DataFrame: A DataFrame showing mismatching timestamps.
    """
    query = f"""
    SELECT 
        local.TrialID,
        local.{timestamp_column} as local_time,
        md.{timestamp_column} as md_time
    FROM 
        {table_name} local
    JOIN 
        md:{table_name} md ON local.TrialID = md.TrialID
    WHERE 
        local.{timestamp_column} != md.{timestamp_column}
    LIMIT 10
    """
    return conn_local.execute(query).df()

def check_null_handling(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str, columns: list) -> dict:
    """
    Check for differences in NULL value handling.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.
        columns (list): List of column names to check.

    Returns:
        dict: A dictionary with column names as keys and mismatch counts as values.
    """
    results = {}
    for column in columns:
        query = f"""
        SELECT 
            COUNT(*) as mismatch_count
        FROM 
            {table_name} local
        FULL OUTER JOIN 
            md:{table_name} md ON local.TrialID = md.TrialID
        WHERE 
            (local.{column} IS NULL AND md.{column} IS NOT NULL) OR
            (local.{column} IS NOT NULL AND md.{column} IS NULL)
        """
        mismatch_count = conn_local.execute(query).fetchone()[0]
        results[column] = mismatch_count
    return results

def compare_query_plans(conn_local: duckdb.DuckDBPyConnection, conn_md: duckdb.DuckDBPyConnection, table_name: str) -> tuple:
    """
    Compare query plans between local and MotherDuck databases.

    Args:
        conn_local (duckdb.DuckDBPyConnection): Connection to the local database.
        conn_md (duckdb.DuckDBPyConnection): Connection to the MotherDuck database.
        table_name (str): Name of the table to compare.

    Returns:
        tuple: A tuple containing the local and MotherDuck query plans.
    """
    query = f"EXPLAIN SELECT * FROM {table_name} WHERE TrialID = 1"
    local_plan = conn_local.execute(query).fetchall()
    md_plan = conn_md.execute(query).fetchall()
    return local_plan, md_plan

In [11]:
# | echo: false

# IN PROGRESS

# def compare_databases_details(local_db: str, md_db: str, table_name) -> bool:
    
#     print(f"Comparing - LOCAL: {local_db} vs MOTHERDUCK: {md_db}")

#     load_dotenv()
#     md_token = os.getenv("MD_TOKEN")
#     os.environ["motherduck_token"] = md_token
#     if not md_token:
#         raise ValueError("MotherDuck token not found in .env file")

#     conn_md = duckdb.connect(f"md:{md_db}?motherduck_token={md_token}")
#     conn_local = duckdb.connect(local_db)
    
#     # Compare table schema
#     schema_diff = compare_table_schema(conn_local, conn_md, table_name)
#     print("Schema differences:")
#     print(schema_diff)
    
#     # Check numeric precision

#     # Attach local database
#     conn.execute(f"ATTACH '{source_db_path}' AS local_source_db")

#     # Attach MotherDuck database

#     conn.execute(f"ATTACH 'md:{md_db_name}?motherduck_token={md_token}' AS md_db")

# # Now call the function
#     precision_diff = check_numeric_precision(conn, 'local_source_db', 'md_db', 'TrialCBs', numeric_columns)
#     numeric_columns = ['TrialNumber', 'CriterionCount']  # Add your numeric column names
#     precision_diff = check_numeric_precision(conn_local, conn_md, table_name, numeric_columns)
#     print("\nNumeric precision differences:")
#     print(precision_diff)
    
#     # Compare timestamps
#     timestamp_diff = compare_timestamps(conn_local, conn_md, table_name, 'TrialStartTime')
#     print("\nTimestamp differences:")
#     print(timestamp_diff)
    
#     # Check NULL handling
#     all_columns = ['TrialID', 'ExperimentID', 'TrialNumber', 'ResponseType', 'TrialSubType', 'CriterionType', 'CriterionCount', 'TrialStartTime', 'TrialEndTime']
#     null_diff = check_null_handling(conn_local, conn_md, table_name, all_columns)
#     print("\nNULL handling differences:")
#     print(null_diff)
    
#     # Compare query plans
#     local_plan, md_plan = compare_query_plans(conn_local, conn_md, table_name)
#     print("\nLocal query plan:")
#     print(local_plan)
#     print("\nMotherDuck query plan:")
#     print(md_plan)
    
#     conn_local.close()
#     conn_md.close()

In [12]:
# | echo: false

# Work in progress (note changed CriterionCount to an integer)

# compare_databases_details(CB_DATABASE, CB_DATABASE_MD, "TrialCBs")